# 5 · The best tile size

The browser does not upscale a frame in one go: it cuts it into **tiles**, runs the model
on each, and joins the results. This notebook trains nothing. It takes the model from
`03_upscale_web.ipynb` and asks what shape to run it at: how big a tile, how many tiles
per run, which upscale factor, and what still works on a weaker GPU.

Three words used throughout:

- **step** - the LR pixels a tile adds to the picture.
- **halo** - the extra pixels around the step that the model needs to see (4 here). The
  model input is the step plus the halo on every side; the halo is computed and thrown away.
- **computed / kept / delivered** pixels - what the GPU computes, what a tile keeps, and
  what the viewer gets. `computed / kept` is the **halo tax**. A tile that does not divide
  the frame hangs over its edge, and that waste is the **tiling tax**.

Frames per second cannot compare two tile sizes (a bigger tile *is* more of a frame), so
speed is measured in **nanoseconds per output pixel**. At 1080p and 60 fps the whole
budget is 8 ns per pixel.

Steps:

1. **Halo tax against tiling tax** - what a square tile wastes, by size.
2. **Rectangles** - tiles that divide the frame exactly, and a check that tiling does not
   change the picture.
3. **The cost of a run** - a fixed cost plus a cost per pixel, fitted to CPU and browser
   timings, and what that means on weaker GPUs.
4. **Batching** - several tiles in one run.
5. **Only what changed** - a desktop rarely changes everywhere at once.
6. **Memory** - what fits in WebGPU's buffers.
7. **The upscale factor** - x2, x3 or x4.
8. **Publish** - every geometry, for the browser to measure.

Needs: `checkpoints/upscale_web.pt` from `03_upscale_web.ipynb`.
Results: `summary.md`, "The geometry a client runs at". The answer is `tiles.TARGET_KEPT_PX`.

In [ ]:
"""Settings."""

import itertools
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from helpers import models, speed, tiles, training, webexport

SWEEP_DIR = training.CHECKPOINT_DIR / "geometry"   # the graphs the CPU sweep times (not published)

STEPS = (32, 64, 96, 128, 160, 192, 256, 320, 384, 512)   # square LR steps to time on the CPU
BATCHES = (1, 2, 4, 8, 16)                                 # tiles per run
BATCH_STEPS = (64, 128)                                    # the steps the batches are timed at
SCALES = (2, 3, 4)                                         # upscale factors

FRAMES = {"720p": (1280, 720), "1080p": (1920, 1080), "1440p": (2560, 1440), "4K": (3840, 2160)}
FRAME = FRAMES["1080p"]                                    # the frame every cost below is for
BUDGET_NS_PER_PIXEL = speed.FRAME_60FPS_MS * 1e6 / (FRAME[0] * FRAME[1])   # one 60 fps frame, per pixel

# Published to dist/upscale/ at the end, for the browser to measure
PUBLISH_STEPS = (64, 128, 192, 256, 384)
PUBLISH_BATCH_STEP = 128
PUBLISH_BATCHES = (4, 16)
PUBLISH_SCALES = (3, 4)
PUBLISH_EXACT_KEPT = ((270, 384), (540, 384), (360, 640), (540, 640))   # (height, width) kept, dividing 1080p

print(f"frame budget: {BUDGET_NS_PER_PIXEL:.2f} ns per output pixel at 1080p60")

In [ ]:
"""The trained model from 03_upscale_web.ipynb."""

checkpoint = training.load_checkpoint(training.CHECKPOINT_DIR / "upscale_web.pt", "cpu")


def build(scale=2):
    """Notebook 3's model at x2, x3 or x4. At x2 every trained weight loads. At x3 and x4 the
    skip is exact but the tail has another shape and stays at zero: right for timing, says
    nothing about quality."""
    model = models.PlainUpscaler(tuple(checkpoint["widths"]), checkpoint["stem"], checkpoint["skip"], scale)
    own = model.state_dict()
    fitting = {key: value for key, value in checkpoint["state_dict"].items() if value.shape == own[key].shape}
    model.load_state_dict(fitting, strict=False)
    return model.eval()


model = build(2)
HALO = model.halo
print(f"widths {checkpoint['widths']}, halo {HALO}, {models.count_parameters(model):,} parameters, "
      f"trained {checkpoint['trained']}")


def geometry(step, scale=2, batch=1):
    """Every size of one run of this model (see `tiles.geometry`)."""
    return tiles.geometry(step, HALO, scale, batch)


def wxh(shape):
    """A (height, width) pair as "WxH"."""
    return f"{shape[1]}x{shape[0]}"

## 1. Halo tax against tiling tax

A bigger tile wastes less on its halo, but a tile that does not divide the frame wastes
its overhang. The two multiply: `total` is what a 1080p frame computes for every pixel it
delivers.

In [ ]:
print(f"{'step':>6}{'input':>10}{'kept':>10}{'halo tax':>10}{'tiles':>7}{'tiling tax':>12}{'total':>8}")
for step in STEPS:
    shape = geometry(step)
    cover = tiles.tiling(FRAME, shape["kept_hr"])
    print(f"{step:>6}{wxh(shape['input'][2:]):>10}{wxh(shape['kept_hr']):>10}{shape['halo_tax']:>9.3f}x"
          f"{cover['tiles']:>7}{cover['tiling_tax']:>11.3f}x{shape['halo_tax'] * cover['tiling_tax']:>7.3f}x")

## 2. Tiles do not have to be square

No square tile divides 1920x1080 at a useful size, but rectangles do: 384x270, 640x360,
640x540... **640x360 divides every 16:9 frame** (720p is 2x2, 1080p 3x3, 1440p 4x4,
4K 6x6), so one graph serves them all. `tiles.plan_tiling` picks the exact tiling nearest
the target size.

In [ ]:
for name, frame in FRAMES.items():
    exact = [row for row in tiles.exact_tilings(frame, halo=HALO) if 128 ** 2 <= row["kept_px"] <= 640 ** 2]
    plan = tiles.plan_tiling(frame, halo=HALO)
    print(f"{name:>6}: {len(exact):3} exact tilings of a useful size; "
          f"chosen {wxh(plan['kept_hr'])} out, {plan['cols']}x{plan['rows']} tiles, "
          f"input {wxh(plan['input'][2:])}{'  (padded)' if plan['padded'] else ''}")

plan = tiles.plan_tiling(FRAME, halo=HALO)   # the 1080p plan, used from here on

### Tiling must not change the picture

With a halo as wide as the model's reach, a frame built from tiles is identical to the
frame run whole. One pixel less halo, and seams appear.

In [ ]:
def upscale_tiled(frame_lr, step, halo, scale=2):
    """Run an LR frame through the model tile by tile, and join the kept centres."""
    step_h, step_w = step
    padded = F.pad(frame_lr, (halo,) * 4, mode="replicate")
    height, width = frame_lr.shape[2:]
    out = torch.zeros(1, 3, height * scale, width * scale)
    for top in range(0, height, step_h):
        for left in range(0, width, step_w):
            tile = padded[:, :, top:top + step_h + 2 * halo, left:left + step_w + 2 * halo]
            with torch.no_grad():
                result = model(tile)
            kept = result[:, :, halo * scale:(halo + step_h) * scale, halo * scale:(halo + step_w) * scale]
            out[:, :, top * scale:(top + step_h) * scale, left * scale:(left + step_w) * scale] = kept
    return out


step_h, step_w = plan["step"]
frame_lr = torch.rand(1, 3, 2 * step_h, 2 * step_w)   # a frame of 2x2 tiles
with torch.no_grad():
    whole = model(frame_lr)
edge = 2 * HALO * 2   # the two runs pad the frame's own border differently, so compare the inside
for halo in (HALO, HALO - 1):
    tiled = upscale_tiled(frame_lr, plan["step"], halo)
    difference = float((tiled - whole)[..., edge:-edge, edge:-edge].abs().max())
    print(f"halo {halo}: largest difference {difference:.1e}  "
          f"{'identical' if difference < 1e-5 else 'SEAMS - the halo is too small'}")

## 3. A run = a fixed cost + a cost per pixel

    ms per run = a + b x computed pixels

`a` is paid once per run, whatever its size: issuing the call, binding buffers, one GPU
submit. `b` is the model's work per pixel. A big `a` favours big tiles; a weaker GPU has
a bigger `b`. The square steps are timed here on the CPU; the browser numbers come from
the benchmark page. A line through each set gives its `a` and `b`.

In [ ]:
"""The square steps on this CPU, priced for a whole 1080p frame."""


def price(shape, ms, frame=FRAME):
    """What a geometry that takes `ms` per run costs for a whole frame."""
    cover = tiles.tiling(frame, shape["kept_hr"], shape["batch"])
    return {**shape, "ms": ms, "runs": cover["runs"], "frame_ms": ms * cover["runs"],
            "ns_per_kept": ms * 1e6 / shape["kept"],
            "ns_per_delivered": ms * cover["runs"] * 1e6 / cover["delivered"]}


def measure(step, batch=1, scale=2):
    """Export one geometry (once), time it on the CPU, and price a frame with it."""
    shape = geometry(step, scale, batch)
    step_h, step_w = shape["step"]
    path = SWEEP_DIR / f"geom_x{scale}_s{step_w}x{step_h}_b{batch}.onnx"
    if not path.exists():
        webexport.export(build(scale), path.name, shape["input"][2:], halo=HALO, batch=batch, models_dir=SWEEP_DIR)
    return price(shape, webexport.time_graph(path, shape["input"][2:], batch=batch))


sweep = [measure(step) for step in STEPS]
print(f"{'step':>6}{'ms/run':>9}{'ns/kept px':>12}{'ns/frame px':>13}{'runs':>6}{'frame ms':>10}")
for row in sweep:
    print(f"{row['step'][0]:>6}{row['ms']:>9.3f}{row['ns_per_kept']:>12.2f}{row['ns_per_delivered']:>13.2f}"
          f"{row['runs']:>6}{row['frame_ms']:>10.1f}")

In [ ]:
"""The browser's numbers, priced the same way."""

# Measured on the benchmark page: NVIDIA Ampere, Chromium, WebGPU, fp16, GPU-resident,
# graph capture. Read differences under 10% as noise.
BROWSER_MS = [   # (scale, step as (height, width), batch, ms per run)
    (2, (64, 64), 1, 0.186),
    (2, (128, 128), 1, 0.293),
    (2, (192, 192), 1, 0.516),
    (2, (256, 256), 1, 0.893),
    (2, (384, 384), 1, 1.721),
    (2, (135, 192), 1, 0.420),   # exact 1080p tilings
    (2, (270, 192), 1, 0.789),
    (2, (180, 320), 1, 0.755),
    (2, (270, 320), 1, 1.175),
    (2, (128, 128), 4, 0.850),   # batches
    (2, (128, 128), 16, 3.040),
    (2, (540, 960), 1, 5.703),   # the whole frame in one run
    (3, (128, 128), 1, 0.446),   # other scales
    (4, (128, 128), 1, 0.463),
]
browser = [price(geometry(step, scale, batch), ms) for scale, step, batch, ms in BROWSER_MS]

print(f"{'scale':>5}{'step':>10}{'keeps':>10}{'batch':>6}{'ms/run':>8}{'ns/kept':>9}{'ns/frame':>10}{'runs':>6}{'1080p':>9}")
for row in sorted(browser, key=lambda row: (row["scale"], row["ns_per_delivered"])):
    print(f"x{row['scale']:<4}{wxh(row['step']):>10}{wxh(row['kept_hr']):>10}{row['batch']:>6}"
          f"{row['ms']:>8.3f}{row['ns_per_kept']:>9.2f}{row['ns_per_delivered']:>10.2f}{row['runs']:>6}"
          f"{row['frame_ms']:>6.1f} ms")

best = min((row for row in browser if row["scale"] == 2 and row["runs"] > 1), key=lambda row: row["ns_per_delivered"])
one_run = min((row for row in browser if row["runs"] == 1), key=lambda row: row["ns_per_delivered"])
print(f"\ncheapest tiling: {wxh(best['kept_hr'])} at {best['ns_per_delivered']:.2f} ns/px "
      f"({best['frame_ms']:.1f} ms a frame, {best['runs']} runs); whole frame: {one_run['ns_per_delivered']:.2f} ns/px")

In [ ]:
"""a and b, fitted to each set of timings."""


def fit(rows):
    """A straight line through (computed pixels, ms per run): a in ms, b in ns per computed pixel."""
    slope, intercept = np.polyfit([row["computed"] for row in rows], [row["ms"] for row in rows], 1)
    return {"fixed_ms": float(intercept), "ns_per_computed": float(slope) * 1e6}


def run_ms(shape, machine):
    """The ms one run of a geometry takes on a fitted machine."""
    return machine["fixed_ms"] + machine["ns_per_computed"] * shape["computed"] / 1e6


# The same card in an older, quieter run (summary.md): ms per run, by square step
OLDER_RUN_MS = {64: 0.155, 128: 0.202, 192: 0.311, 256: 0.452, 320: 0.657}

fits = {
    "this CPU": fit(sweep),
    "Ampere (browser)": fit([row for row in browser if row["scale"] == 2 and row["batch"] == 1]),
    "Ampere (older run)": fit([{**geometry(step), "ms": ms} for step, ms in OLDER_RUN_MS.items()]),
}
for name, machine in fits.items():
    print(f"{name:<20} a = {machine['fixed_ms']:.4f} ms per run, b = {machine['ns_per_computed']:.4f} ns per pixel")

ampere = fits["Ampere (browser)"]
spread = ampere["ns_per_computed"] / fits["Ampere (older run)"]["ns_per_computed"]
crossover = ampere["fixed_ms"] * 1e6 / ampere["ns_per_computed"]   # in computed HR pixels
print(f"\nThe older run was {spread:.1f}x faster per pixel on the same card, so absolute numbers are a band "
      f"that wide.\nOn the Ampere card one run's fixed cost equals {crossover:,.0f} computed pixels "
      f"(a step of about {math.sqrt(crossover) / 2:.0f} LR pixels). Smaller tiles are mostly overhead.")

### On weaker GPUs

Every other GPU is a spec sheet here: it gets the Ampere card's fixed cost `a`, and its
`b` scaled by its TFLOPS (the measured card taken as an RTX 3060). Crude, but enough to see
the shape. Its share of the 1080p60 budget is a band, from the older run's speed to today's.

In [ ]:
"""ns per delivered pixel of a 1080p frame, in 640x360 tiles and in one run, on each GPU."""

# Rough vendor figures, fp32 TFLOPS - used for ratios only
GPU_TFLOPS = {"Iris Xe 96EU": 1.7, "Apple M1 8-core": 2.6, "GTX 1650": 2.9, "RTX 3050 laptop": 5.5,
              "RX 6600": 8.9, "RTX 3060": 12.7}
MEASURED_AS = "RTX 3060"                        # what the measured Ampere card is taken to be
whole_step = (FRAME[1] // 2, FRAME[0] // 2)     # the whole frame as one tile


def frame_ns(step, machine):
    """ns per delivered pixel of a 1080p frame cut into tiles of `step`, on a fitted machine."""
    shape = geometry(step)
    cover = tiles.tiling(FRAME, shape["kept_hr"])
    return run_ms(shape, machine) * cover["runs"] * 1e6 / cover["delivered"]


gpus = {name: {"fixed_ms": ampere["fixed_ms"],
               "ns_per_computed": ampere["ns_per_computed"] * GPU_TFLOPS[MEASURED_AS] / tflops}
        for name, tflops in GPU_TFLOPS.items()}

plan_tiles = f"{wxh(plan['kept_hr'])} tiles"
print(f"{'GPU':<18}{plan_tiles:>16}{'whole frame':>14}{'of the 1080p60 budget':>25}")
for name, machine in gpus.items():
    tiled_ns, whole_ns = frame_ns(plan["step"], machine), frame_ns(whole_step, machine)
    print(f"{name:<18}{tiled_ns:>13.2f} ns{whole_ns:>11.2f} ns"
          f"{tiled_ns / spread / BUDGET_NS_PER_PIXEL:>17.0%} - {tiled_ns / BUDGET_NS_PER_PIXEL:.0%}")

steps = range(32, 561, 8)
plt.figure(figsize=(9, 4))
for name, machine in gpus.items():
    plt.plot(steps, [frame_ns(step, machine) for step in steps], label=name)
plt.axhline(BUDGET_NS_PER_PIXEL, color="black", linestyle="--", linewidth=1, label="1080p60 budget")
plt.axvline(plan["step"][1], color="grey", linestyle=":", label=f"{wxh(plan['kept_hr'])} tiles")
plt.xlabel("square LR step (pixels)")
plt.ylabel("ns per delivered pixel")
plt.yscale("log")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

Bigger is always cheaper on this model: the curves have no minimum inside, and the
cheapest is one run for the whole frame. But a remote desktop usually changes in a small
area, and a tile that did not change can be skipped - a bigger tile skips less. So the
limit on tile size is not speed: it is **what can be skipped** (section 5), **what fits in
memory** (section 6), and keeping the GPU free for the rest of the page.

## 4. Batching

A batch (N tiles in one run) also saves fixed costs, like a bigger tile. It pays the halo
N times, but keeps small tiles, so unchanged tiles can still be skipped.

In [ ]:
"""Batches on the CPU, then on the Ampere card: what the fit predicts and what the page measured."""

batch_rows = [measure(step, batch) for step, batch in itertools.product(BATCH_STEPS, BATCHES)]
print(f"{'step':>6}{'batch':>6}{'ms/run':>9}{'ns/kept px':>12}")
for row in batch_rows:
    print(f"{row['step'][0]:>6}{row['batch']:>6}{row['ms']:>9.3f}{row['ns_per_kept']:>12.2f}")

step = PUBLISH_BATCH_STEP
measured = {row["batch"]: row["ms"] for row in browser if row["scale"] == 2 and row["step"] == (step, step)}
print(f"\nAmpere, step {step}:  {'batch':>5}{'predicted':>11}{'measured':>10}{'ms/tile':>9}")
for batch in BATCHES:
    predicted = run_ms(geometry(step, batch=batch), ampere)
    seen = measured.get(batch)
    print(f"{'':<20}{batch:>5}{predicted:>11.3f}{(f'{seen:.3f}' if seen else '-'):>10}"
          f"{(seen or predicted) / batch:>9.3f}")

## 5. Only some of the screen changes

On a desktop, a moving cursor or a line of typed text changes a small area. Only the tiles
it touches need to run, so the best tile size depends on *what* moved.

In [ ]:
"""ms per 1080p frame on the Ampere card, running only the tiles a change touches."""

CHANGES = {   # what changed on screen: (width, height) in output pixels
    "cursor, 32x32": (32, 32),
    "a line of text, 400x30": (400, 30),
    "a dragged window, 800x600": (800, 600),
    "video, 1280x720": (1280, 720),
    "everything": FRAME,
}
CHANGE_STEPS = (64, 128, 192, 256, 320, 384)


def touched_tiles(step, region):
    """How many tiles a changed (width, height) region at a random position touches, on
    average: a region w wide spans w / tile width + 1 columns, and never more than there are."""
    kept_h, kept_w = geometry(step)["kept_hr"]
    cover = tiles.tiling(FRAME, (kept_h, kept_w))
    return math.ceil(min(cover["cols"], region[0] / kept_w + 1) * min(cover["rows"], region[1] / kept_h + 1))


print(f"{'what changed':<26}" + "".join(f"{step:>8}" for step in CHANGE_STEPS) + "   (by step)")
for name, region in CHANGES.items():
    print(f"{name:<26}" + "".join(f"{run_ms(geometry(step), ampere) * touched_tiles(step, region):>8.2f}"
                                   for step in CHANGE_STEPS))

whole_ms = run_ms(geometry(whole_step), ampere)
tiled_ms = plan["runs"] * run_ms(geometry(plan["step"]), ampere)
print(f"\nwhole frame in one run: {whole_ms:.2f} ms; all {plan['runs']} tiles of {wxh(plan['kept_hr'])}: "
      f"{tiled_ms:.2f} ms")
print(f"so tiling wins while fewer than {whole_ms / tiled_ms:.0%} of tiles change")

## 6. Memory

The widest tensor is the 32-channel body at the input size. It grows with tile area and
batch, and WebGPU has default buffer limits (128 MiB per binding, 256 MiB per buffer).

In [ ]:
"""The tensors of one float16 run, by step and batch, against WebGPU's default limits."""

MAX_BINDING_BYTES = 128 * 2**20   # WebGPU's defaults for one buffer binding,
MAX_BUFFER_BYTES = 256 * 2**20    # and for one buffer, unless the page asks for more


def working_set(step, batch=1, scale=2, bytes_per_value=2):
    """The largest tensor of one run, and all of them together, in bytes."""
    _, _, height, width = geometry(step, scale, batch)["input"]
    channel = batch * height * width * bytes_per_value   # one channel at the input's size
    tensors = [3 * channel,                          # the input
               max(checkpoint["widths"]) * channel,  # the widest layer
               3 * scale**2 * channel,               # the output convolution, before PixelShuffle
               3 * scale**2 * channel]               # the output: 3 colours at scale² the area
    return max(tensors), sum(tensors)


print(f"{'step':>6}{'batch':>6}{'largest':>12}{'total':>12}")
for step, batch in itertools.product((128, 256, 512), (1, 4, 16)):
    largest, total = working_set(step, batch)
    flags = ("  over the binding limit" if largest > MAX_BINDING_BYTES else "") + \
            ("  over the buffer limit" if total > MAX_BUFFER_BYTES else "")
    print(f"{step:>6}{batch:>6}{largest / 2**20:>8.1f} MiB{total / 2**20:>8.1f} MiB{flags}")

## 7. The upscale factor

All convolutions run at low resolution, so the work per *output* pixel falls with the
square of the scale: x4 does a quarter of x2's work per output pixel, and the stream sends
a quarter of the pixels. The cost is quality - and the data here is x2 only, so the x3 and
x4 models are untrained and **no quality is claimed** for them.

In [ ]:
"""Work and time per run at each scale, step 128: on the CPU and in the browser."""

print(f"{'scale':>6}{'MACs/out px':>13}{'CPU ms/run':>12}{'browser ms/run':>16}{'browser ns/frame px':>21}")
for scale in SCALES:
    cpu = measure(128, scale=scale)
    seen = next(row for row in browser if row["scale"] == scale and row["step"] == (128, 128) and row["batch"] == 1)
    print(f"x{scale:<5}{speed.macs_per_lr_pixel(build(scale)) / scale ** 2:>13,.0f}{cpu['ms']:>12.3f}"
          f"{seen['ms']:>16.3f}{seen['ns_per_delivered']:>21.2f}")

## 8. Publish the family

Every geometry as a float16 graph in `dist/upscale/`, for the benchmark page to measure on
real GPUs. Put new browser numbers back into `BROWSER_MS` (section 3) and run again.

In [ ]:
def publish(step, batch=1, scale=2, note=""):
    """Write one geometry as a float16 graph to dist/upscale/."""
    shape = geometry(step, scale, batch)
    step_h, step_w = shape["step"]
    label = f"geom x{scale} step {step_w}x{step_h}" + (f" batch {batch}" if batch > 1 else "") + \
            (f" - {note}" if note else "")
    name = f"upscale_geom_x{scale}_s{step_w:04d}x{step_h:04d}_b{batch:02d}.onnx"
    fp32 = webexport.export(build(scale), name, shape["input"][2:], label=f"{label} fp32", halo=HALO,
                            batch=batch, models_dir=SWEEP_DIR)
    fp16 = webexport.to_float16(fp32, name)
    webexport.write_metadata(fp16["path"], {"label": f"{label} fp16"})
    print(webexport.summary_line(fp16))


for step in PUBLISH_STEPS:
    publish(step)
for batch in PUBLISH_BATCHES:
    publish(PUBLISH_BATCH_STEP, batch=batch)
for scale in PUBLISH_SCALES:
    publish(128, scale=scale)
for kept in PUBLISH_EXACT_KEPT:
    cover = tiles.tiling(FRAME, kept)
    publish((kept[0] // 2, kept[1] // 2), note=f"exact 1080p, {cover['cols']}x{cover['rows']} tiles")

print("\nmeasure them in a browser: uv run src/upscale/benchmark/main.py")

## The answer

**640x360 output tiles: a 320x180 step, a 328x188 model input.** It was the cheapest
tiling measured (about 6.8 ms for a 1080p frame in 9 runs), and it divides every 16:9
resolution exactly. `tiles.TARGET_KEPT_PX` is set to it.

- The old 128 step costs about 70% more per frame: 40 runs, each with a fixed cost.
- Rectangles that fit the frame beat squares that do not.
- One run for the whole frame is cheaper still, but cannot skip unchanged tiles.
- Batching helps, less than predicted.
- x4 is half the cost per output pixel of x2, but its quality is not measured yet.
- On a mid-low GPU, plan for 720p60 or 1080p30.

The full reasoning and numbers are in `summary.md`.